# U-Net 学習（Google Colab）— 骨格モード

**上から順に「すべてのセルを実行」するだけで、バッキングとリードの両方（または設定した方だけ）が
学習できるように構成されている。** 「学習設定」セクションのトグルで実行対象を選ぶ。

## バッキング（`TRAIN_BACKING`）
1. **パッチ化** … 合成 15,000 + Guitar-TECHS（**骨格 → フル** ペア）
2. **Stage 1** … 合成データで学習
3. **Stage 2（任意: `RUN_STAGE2`）** … Guitar-TECHS でファインチューニングし push
4. **Stage 1.5（任意: `RUN_STAGE1_5`）** … onset 重み強化 fine-tune（ストローク増加・小節末多様化）。
   Stage1 から fine-tune し `checkpoints/stage1_h2/unet_last.pt` を push

## リード（`TRAIN_LEAD`）
- 進行のコードトーン骨格 → 単音リードを学習（バッキングと完全に独立、`checkpoints/lead/unet_last.pt` を push）

**学習タスク（バッキング用セル）:** 既定は `downbeat_chord`（小節頭コード骨格 → リッチなギターパート）。  
旧 `onset_to_full` チェックポイントとは **別タスク** のため、骨格用には **再学習が必要** です。

**事前設定（セル 1）**
- リポジトリが **Private** の場合: `GITHUB_TOKEN`（PAT, repo 権限）が **clone 時から必須**
- リポジトリが **Public** の場合: `GITHUB_TOKEN` は checkpoint push 時のみ必要

In [ ]:
REPO_URL = "https://github.com/Kakeru0307/colab_train.git"
GITHUB_TOKEN = ""  # ← Private リポジトリならここに PAT（repo 権限）を入れる

import os
if GITHUB_TOKEN:
    os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN

def authenticated_repo_url(repo_url: str, token: str) -> str:
    if not token:
        return repo_url
    if repo_url.startswith("https://") and "@" not in repo_url:
        return repo_url.replace("https://", f"https://{token}@")
    return repo_url

CLONE_URL = authenticated_repo_url(REPO_URL, GITHUB_TOKEN)
print("clone:", REPO_URL)
print("token:", "set" if GITHUB_TOKEN else "not set")

In [ ]:
import os
import shutil
from pathlib import Path

REPO_DIR = Path("repo")

# 作業ディレクトリを /content に戻す（再実行時の安全策）
if not Path("train.py").exists() and Path("/content").exists():
    os.chdir("/content")

if Path("train.py").exists():
    print("既にリポジトリ内です:", os.getcwd())
else:
    if REPO_DIR.exists() and not (REPO_DIR / "train.py").exists():
        print("不完全な repo を削除します")
        shutil.rmtree(REPO_DIR)

    if REPO_DIR.exists() and (REPO_DIR / "train.py").exists():
        print("既存の repo を使用します")
        os.chdir(REPO_DIR)
    else:
        if not GITHUB_TOKEN:
            print("警告: Private リポジトリの場合、セル1で GITHUB_TOKEN を設定してください")
        !git clone {CLONE_URL} repo
        os.chdir("repo")

    if not Path("train.py").exists():
        raise FileNotFoundError(
            "clone 失敗。GITHUB_TOKEN を設定するか、ランタイムを再起動して再実行してください。"
        )

print("OK:", os.getcwd())
!ls train.py prepare_all.py colab_train.ipynb

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 学習設定（ここだけ変えれば「すべてのセルを実行」で目的の学習が回る）

下のトグルで **バッキング** と **リード** を独立に on/off できる。両方 `True` にすれば
上から下まで実行するだけで両方の学習が完走する。片方だけ学習したい場合は片方を `False`
にすれば、対応するセルは自動的にスキップされる（データ生成もモデルも完全に独立しているため
順序を気にする必要はない）。

In [ ]:
TRAIN_BACKING = True   # バッキング（Stage1 → Stage2 → Stage1.5）を実行するか
TRAIN_LEAD = True      # リード（単音）学習を実行するか

# バッキング内の任意ステージ（TRAIN_BACKING=True のときのみ意味を持つ）
RUN_STAGE2 = True       # curated Guitar-TECHS での弱微調整（崩れたら False にして Stage1 のまま使う）
RUN_STAGE1_5 = True     # onset 重み強化 fine-tune（ストローク増加・小節末多様化。Stage1 から分岐）

print(f"TRAIN_BACKING={TRAIN_BACKING} (RUN_STAGE2={RUN_STAGE2}, RUN_STAGE1_5={RUN_STAGE1_5})")
print(f"TRAIN_LEAD={TRAIN_LEAD}")

## バッキング学習

`TRAIN_BACKING` が `True` のときのみ以下が実行される（データ作成 → Stage1 → Stage2 → push →
Stage1.5 → push）。`False` の場合は全セルが安全にスキップされ、次の「リード学習」セクションへ進む。

In [ ]:
if TRAIN_BACKING:
    SYNTHETIC_COUNT = 15000
    SYNTHETIC_SEED = 42
    SYNTHETIC_BARS = 8  # 8固定（長くするとパッチ爆発でディスク落ち）
    # downbeat_chord | root_per_bar | melody_line | onset_to_full
    PAIR_MODE = "downbeat_chord"

    # 容量不足で落ちた場合: ランタイム初期化後、下を実行してから本セルへ
    # !rm -rf data/pairs data/raw/synthetic

    !python prepare_all.py \
      --synthetic-count {SYNTHETIC_COUNT} \
      --synthetic-seed {SYNTHETIC_SEED} \
      --bars {SYNTHETIC_BARS} \
      --force-regenerate \
      --mode {PAIR_MODE}
else:
    print("TRAIN_BACKING=False のためスキップ（バッキング用データ作成）")

In [ ]:
if TRAIN_BACKING:
    STAGE1_EPOCHS = 20   # 試作確認用。足りなければ 40 に戻す
    BATCH_SIZE = 16
    POS_WEIGHT = 20      # onsetセルは全体の約0.05%。無音に潰れるのを防ぐ強めの重み

    # Stage1: 合成データで基礎（進行・リズム）を学習。lr=1e-3 でしっかり収束させる
    #   ※ loss が下がらず発散する場合は --lr 5e-4 に下げる
    #   ※ 入力は 12ch（tonal11 + BPM条件1）。旧 in=11 の ckpt は使えない
    !python train.py \
      --pairs-dir data/pairs/synthetic \
      --epochs {STAGE1_EPOCHS} \
      --batch-size {BATCH_SIZE} \
      --lr 1e-3 \
      --checkpoint-dir checkpoints/stage1 \
      --pos-weight {POS_WEIGHT}
else:
    print("TRAIN_BACKING=False のためスキップ（Stage1）")

In [ ]:
if TRAIN_BACKING and RUN_STAGE2:
    # Stage2（弱い微調整）: curated TECHS のみ
    #   P3_music 全件 + P1/P2_chords 各最大6本。全量 TECHS はリズムを崩すので使わない。
    #   lr/epoch は小さく Stage1 の刻みを守る。
    STAGE2_EPOCHS = 5
    STAGE2_LR = 1e-5

    !python train.py \
      --pairs-dir data/pairs/guitar-techs-curated \
      --epochs {STAGE2_EPOCHS} \
      --batch-size {BATCH_SIZE} \
      --lr {STAGE2_LR} \
      --checkpoint-dir checkpoints/stage2 \
      --resume checkpoints/stage1/unet_last.pt \
      --pos-weight {POS_WEIGHT}
else:
    print("TRAIN_BACKING=False または RUN_STAGE2=False のためスキップ（Stage2）")

In [ ]:
if TRAIN_BACKING and RUN_STAGE2:
    import json
    from pathlib import Path

    manifest_path = Path("data/pairs/manifest_all.json")
    pair_mode = json.loads(manifest_path.read_text(encoding="utf-8"))["mode"] if manifest_path.exists() else PAIR_MODE

    !git config user.email "colab@users.noreply.github.com"
    !git config user.name "Colab Train"
    # curated Stage2（弱微調整）後は stage2 を push。リズムが崩れたら stage1 に差し替え
    !python scripts/push_checkpoint.py \
      --ckpt checkpoints/stage2/unet_last.pt \
      --message "Stage2 curated TECHS (P3_music+chords薄) {pair_mode}"
else:
    print("TRAIN_BACKING=False または RUN_STAGE2=False のためスキップ（Stage2 push）")

## Stage1.5: onset 重み強化 fine-tune（ストローク増加・小節末多様化対応）

`TRAIN_BACKING` と `RUN_STAGE1_5`（学習設定セル）が両方 `True` のときのみ実行される。

**症状**: バッキングが1小節あたり1〜2打ちしか出ない／毎小節末が同じ位置（tick14-15）で機械的に空く。

**対応**: `weighted_mse_loss` に `--onset-weight`（onsetセル全般の重み）と `--midbar-onset-bonus`
（小節頭以外のonsetへの追加重み）を追加。**MSEのまま**重みだけ強化する方式（BCE化はしない。
model の出力層に活性化がなく raw 非有界値のため、BCE+clamp は勾配消失・チャンネル間の意味論不一致を
起こす。詳細は `docs/user/handoff.md` 参照）。

**土台は Stage1**（Stage2 ではない）: Stage2 の curated TECHS fine-tune はリズムを崩す傾向がある
という既知の教訓があるため、本 fine-tune は `checkpoints/stage1/unet_last.pt` から行う。

**効果が薄い場合**: `MIDBAR_ONSET_BONUS` を 3.0 → 6.0 に上げて再実行（epochs +2）。
**精度が落ちた場合**（`precision_vs_chord` が下記ベースラインより 10pt 以上低下）: `ONSET_WEIGHT` を 1.5 に下げる。

In [ ]:
if TRAIN_BACKING and RUN_STAGE1_5:
    # ベースライン計測（fine-tune 前の合成 target データの統計）
    # --first-patch-only: 8小節ちょうどの合成曲は patches:2 で重複パッチが出るバグの回避
    !python scripts/analyze_bar_rhythm.py \
      --pairs-dir data/pairs/synthetic \
      --n-samples 200 \
      --first-patch-only
else:
    print("TRAIN_BACKING=False または RUN_STAGE1_5=False のためスキップ（ベースライン計測）")

In [ ]:
if TRAIN_BACKING and RUN_STAGE1_5:
    H2_EPOCHS = 5
    H2_LR = 2e-5          # フル学習(1e-3)の 1/50 程度。fine-tune なので小さく
    ONSET_WEIGHT = 2.0        # onset(値1)セルへの追加重み
    MIDBAR_ONSET_BONUS = 3.0  # 小節頭以外のonsetへの追加重み（中拍の打鍵を強化）

    !python train.py \
      --pairs-dir data/pairs/synthetic \
      --epochs {H2_EPOCHS} \
      --batch-size {BATCH_SIZE} \
      --lr {H2_LR} \
      --checkpoint-dir checkpoints/stage1_h2 \
      --resume checkpoints/stage1/unet_last.pt \
      --pos-weight {POS_WEIGHT} \
      --onset-weight {ONSET_WEIGHT} \
      --midbar-onset-bonus {MIDBAR_ONSET_BONUS}
else:
    print("TRAIN_BACKING=False または RUN_STAGE1_5=False のためスキップ（Stage1.5 fine-tune）")

In [ ]:
if TRAIN_BACKING and RUN_STAGE1_5:
    # fine-tune 後の効果測定: 実際にモデルで推論して onsets/bar・bar_end_entropy を再計測する。
    # （analyze_bar_rhythm.py --pairs-dir は target npy を読むだけで推論しないため、
    #   fine-tune 前後で同じ値になってしまう。ここでは実際に checkpoints/stage1_h2 で
    #   予測させた出力を評価する）
    import random
    from pathlib import Path

    import numpy as np
    import torch

    from model import build_unet
    from midi_to_patch import normalize_pianoroll

    import sys
    sys.path.insert(0, "scripts")
    from analyze_bar_rhythm import aggregate_metrics, print_report  # noqa: E402

    EVAL_N = 200
    EVAL_SEED = 0
    ONSET_TH, SUSTAIN_TH = 0.5, 0.85  # prttype/inference.py の既定値と揃える

    def _denormalize_for_eval(values: np.ndarray) -> np.ndarray:
        q = np.zeros_like(values, dtype=np.uint8)
        q[values >= ONSET_TH] = 1
        q[values >= SUSTAIN_TH] = 2
        return q

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(
        "checkpoints/stage1_h2/unet_last.pt", map_location=device, weights_only=False
    )
    model = build_unet(
        in_channels=ckpt.get("in_channels", 12), out_channels=ckpt.get("out_channels", 11)
    ).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    input_dir = Path("data/pairs/synthetic/input")
    files = sorted(input_dir.rglob("*_tonal.npy"))
    rng = random.Random(EVAL_SEED)
    if len(files) > EVAL_N:
        files = rng.sample(files, EVAL_N)

    samples = []
    with torch.no_grad():
        for f in files:
            raw = np.load(f).astype(np.uint8)  # (11,128,128) skeleton
            cond_path = f.with_name(f.name.replace("_tonal.npy", "_cond.npy"))
            bpm_unit = float(np.load(cond_path)) if cond_path.exists() else 0.5
            model_in = np.concatenate(
                [normalize_pianoroll(raw).astype(np.float32),
                 np.full((1, 128, 128), bpm_unit, dtype=np.float32)],
                axis=0,
            )
            tensor = torch.from_numpy(model_in).unsqueeze(0).to(device)
            out = model(tensor).squeeze(0).cpu().numpy()
            pred = _denormalize_for_eval(out)
            samples.append((pred, raw))  # (モデル予測, 入力骨格). precision は骨格との整合率

    m = aggregate_metrics(samples)
    print_report(f"fine-tune後モデル予測 (checkpoints/stage1_h2, n={len(samples)})", m)
else:
    print("TRAIN_BACKING=False または RUN_STAGE1_5=False のためスキップ（Stage1.5 効果測定）")

In [ ]:
if TRAIN_BACKING and RUN_STAGE1_5:
    !git config user.email "colab@users.noreply.github.com"
    !git config user.name "Colab Train"
    !python scripts/push_checkpoint.py \
      --ckpt checkpoints/stage1_h2/unet_last.pt \
      --message "Stage1.5 onset重み強化 fine-tune (onset_weight={ONSET_WEIGHT}, midbar_onset_bonus={MIDBAR_ONSET_BONUS})"
else:
    print("TRAIN_BACKING=False または RUN_STAGE1_5=False のためスキップ（Stage1.5 push）")

## リード学習

`TRAIN_LEAD`（学習設定セル）が `True` のときのみ実行される。**バッキングと完全に独立**
（データ生成・学習・チェックポイントとも別。`TRAIN_BACKING` の値に関係なく単独で実行可能）。

進行のコードトーン骨格 → **単音リード** を学習する。backing とは別データ・別チェックポイント（`checkpoints/lead`）。
- 入力: backing と共通の骨格（小節頭コードトーン）
- 正解: スケール上の単音リード（`generate_lead_pairs.py` が生成。backing の `data/pairs/synthetic` には依存しない）
- Stage2 は行わない（合成のみ 1 段。backing で Stage2 がリズムを崩した教訓）

In [ ]:
if TRAIN_LEAD:
    # リード用 input/target ペアを生成（進行のコード骨格 → 単音リード）
    LEAD_COUNT = 6000
    !python generate_lead_pairs.py \
      --pairs-dir data/pairs/lead \
      --count {LEAD_COUNT} \
      --seed 42
else:
    print("TRAIN_LEAD=False のためスキップ（リード用データ作成）")

In [ ]:
if TRAIN_LEAD:
    # リード学習（合成のみ 1 段。backing と同じ設定で収束させる）
    LEAD_EPOCHS = 20
    LEAD_BATCH_SIZE = 16
    LEAD_POS_WEIGHT = 20
    !python train.py \
      --pairs-dir data/pairs/lead \
      --epochs {LEAD_EPOCHS} \
      --batch-size {LEAD_BATCH_SIZE} \
      --lr 1e-3 \
      --checkpoint-dir checkpoints/lead \
      --pos-weight {LEAD_POS_WEIGHT}
else:
    print("TRAIN_LEAD=False のためスキップ（リード学習）")

In [ ]:
if TRAIN_LEAD:
    # リードのチェックポイントを push（ローカルでは checkpoints/lead/ に配置）
    !git config user.email "colab@users.noreply.github.com"
    !git config user.name "Colab Train"
    !python scripts/push_checkpoint.py \
      --ckpt checkpoints/lead/unet_last.pt \
      --message "lead synthetic (単音リード)"
else:
    print("TRAIN_LEAD=False のためスキップ（リード push）")

## 実行サマリ

In [ ]:
from pathlib import Path

print("=== 学習設定 ===")
print(f"TRAIN_BACKING={TRAIN_BACKING}  RUN_STAGE2={RUN_STAGE2}  RUN_STAGE1_5={RUN_STAGE1_5}")
print(f"TRAIN_LEAD={TRAIN_LEAD}")
print()
print("=== 生成されたチェックポイント ===")
for label, path in [
    ("backing Stage1   ", "checkpoints/stage1/unet_last.pt"),
    ("backing Stage2   ", "checkpoints/stage2/unet_last.pt"),
    ("backing Stage1.5 ", "checkpoints/stage1_h2/unet_last.pt"),
    ("lead             ", "checkpoints/lead/unet_last.pt"),
]:
    exists = Path(path).exists()
    print(f"  [{'OK' if exists else '--'}] {label}: {path}")